In [1]:

# ==============================
# TASK 1: DATA ACQUISITION & EXPLORATION
# ==============================

!pip -q install wfdb

import os
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Download MIT-BIH Arrhythmia Database
DATA_DIR = "/content/mitdb"
os.makedirs(DATA_DIR, exist_ok=True)

wfdb.dl_database("mitdb", dl_dir=DATA_DIR)

# Get all record names
RECORDS = sorted(wfdb.get_record_list("mitdb"))

# Remove paced records
PACED_RECORDS = ['102', '104', '107', '217']
RECORDS = [r for r in RECORDS if r not in PACED_RECORDS]

FS = 360

# AAMI class mapping
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    '/': 'Q', 'f': 'Q', 'Q': 'Q', '?': 'Q'
}

CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q']

print("Total recordings after removing paced records:", len(RECORDS))
print("Sampling frequency:", FS, "Hz")

# Inspect one ECG recording
sample_record = wfdb.rdrecord(os.path.join(DATA_DIR, '100'))
sample_annotation = wfdb.rdann(os.path.join(DATA_DIR, '100'), 'atr')

ecg = sample_record.p_signal[:, 0]

# Plot first 10 seconds of ECG
plt.figure(figsize=(15, 4))
time = np.arange(10 * FS) / FS

plt.plot(time, ecg[:10 * FS], label="ECG Signal")
plt.title("MIT-BIH ECG Signal - Record 100")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.legend()
plt.grid()
plt.show()

# Plot ECG with beat annotations
plt.figure(figsize=(15, 4))
plt.plot(time, ecg[:10 * FS], label="ECG")

for pos, symbol in zip(sample_annotation.sample,
                       sample_annotation.symbol):
    if pos < 10 * FS and symbol in AAMI_MAP:
        plt.scatter(pos / FS, ecg[pos], color='red', s=20)

plt.title("ECG Signal with Beat Annotations")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.grid()
plt.legend()
plt.show()

# Check AAMI class distribution in all selected recordings
class_counts = Counter()

for rec in RECORDS:
    ann = wfdb.rdann(os.path.join(DATA_DIR, rec), 'atr')

    for symbol in ann.symbol:
        if symbol in AAMI_MAP:
            class_counts[AAMI_MAP[symbol]] += 1

distribution = pd.Series(
    {c: class_counts[c] for c in CLASS_NAMES}
)

print("\nAAMI Class Distribution:")
print(distribution)

plt.figure(figsize=(8, 4))
sns.barplot(x=distribution.index, y=distribution.values)
plt.title("AAMI Arrhythmia Class Distribution")
plt.xlabel("AAMI Classes")
plt.ylabel("Number of Beats")
plt.show()


Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

NetFileError: 502 Error: Bad Gateway for url: https://physionet.org/files/mitdb/1.0.0/104.atr

In [2]:

# ==============================
# TASK 2: DATA PREPROCESSING
# ==============================

from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import GroupShuffleSplit

# Standard inter-patient division
DS1 = [
    '101','106','108','109','112','114','115','116',
    '118','119','122','124','201','203','205','207',
    '208','209','215','220','223','230'
]

DS2 = [
    '100','103','105','111','113','117','121','123',
    '200','202','210','212','213','214','219','221',
    '222','228','231','232','233','234'
]

# Beat segmentation settings
BEFORE = int(0.2 * FS)    # 0.2 seconds before R-peak
AFTER = int(0.4 * FS)     # 0.4 seconds after R-peak
WINDOW_SIZE = BEFORE + AFTER

# Create bandpass filter
sos = butter(
    4, [0.5, 40],
    btype='bandpass',
    fs=FS,
    output='sos'
)

# Convert AAMI labels into numerical values
LABEL_TO_INT = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}

# Function to process ECG recordings
def process_records(record_list):

    beats = []
    labels = []
    groups = []

    for rec in record_list:

        # Load ECG signal and annotations
        record = wfdb.rdrecord(os.path.join(DATA_DIR, rec))
        annotation = wfdb.rdann(
            os.path.join(DATA_DIR, rec), 'atr'
        )

        # Select first ECG channel
        signal = record.p_signal[:, 0]

        # Remove baseline wander and noise
        filtered_signal = sosfiltfilt(sos, signal)

        # Extract individual beats
        for peak, symbol in zip(
            annotation.sample, annotation.symbol
        ):

            # Ignore non-beat annotations
            if symbol not in AAMI_MAP:
                continue

            # Skip beats near recording boundaries
            start = peak - BEFORE
            end = peak + AFTER

            if start < 0 or end > len(filtered_signal):
                continue

            beat = filtered_signal[start:end]

            # Z-score normalization
            mean = np.mean(beat)
            std = np.std(beat)

            if std < 1e-8:
                continue

            beat = (beat - mean) / std

            beats.append(beat.astype(np.float32))
            labels.append(LABEL_TO_INT[AAMI_MAP[symbol]])
            groups.append(rec)

        print(f"Processed record: {rec}")

    return (
        np.array(beats, dtype=np.float32)[..., np.newaxis],
        np.array(labels, dtype=np.int64),
        np.array(groups)
    )


# Process training/validation records
X_ds1, y_ds1, groups_ds1 = process_records(DS1)

# Process independent test records
X_test, y_test, groups_test = process_records(DS2)

print("\nDataset shapes:")
print("DS1:", X_ds1.shape, y_ds1.shape)
print("DS2:", X_test.shape, y_test.shape)

# Split DS1 into training and validation by patient
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X_ds1, y_ds1, groups=groups_ds1)
)

X_train, y_train = X_ds1[train_idx], y_ds1[train_idx]
X_val, y_val = X_ds1[val_idx], y_ds1[val_idx]

print("\nFinal Data Split:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nTraining patients:", np.unique(groups_ds1[train_idx]))
print("Validation patients:", np.unique(groups_ds1[val_idx]))
print("Testing patients:", np.unique(groups_test))

# Display training class distribution
train_distribution = pd.Series(y_train).map(
    {v: k for k, v in LABEL_TO_INT.items()}
).value_counts().reindex(CLASS_NAMES, fill_value=0)

print("\nTraining Class Distribution:")
print(train_distribution)

plt.figure(figsize=(8, 4))
sns.barplot(
    x=train_distribution.index,
    y=train_distribution.values
)
plt.title("Training Data Class Distribution")
plt.xlabel("AAMI Classes")
plt.ylabel("Number of Beats")
plt.show()


NameError: name 'FS' is not defined

In [ ]:

# ==============================
# TASK 3: RNN, LSTM AND GRU TRAINING
# ==============================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, SimpleRNN, LSTM, GRU,
    Dense, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Calculate class weights to handle imbalanced data
unique_classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)

class_weights = {
    int(c): float(w)
    for c, w in zip(unique_classes, weights)
}

print("Class Weights:", class_weights)

# Function to create recurrent models
def build_model(model_type):

    model = Sequential([
        Input(shape=(WINDOW_SIZE, 1)),

        model_type(64),

        Dropout(0.3),

        Dense(32, activation='relu'),

        Dropout(0.2),

        Dense(5, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


# Three recurrent architectures
model_types = {
    'RNN': SimpleRNN,
    'LSTM': LSTM,
    'GRU': GRU
}

models = {}
histories = {}

# Train all three models with identical settings
for name, layer in model_types.items():

    print("\n" + "=" * 50)
    print("Training", name, "Model")
    print("=" * 50)

    tf.keras.utils.set_random_seed(42)

    model = build_model(layer)

    model.summary()

    # Early stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=25,
        batch_size=256,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=1
    )

    models[name] = model
    histories[name] = history

    print(name, "training completed!")

print("\nAll three models have been trained successfully.")


In [ ]:

# ==============================
# TASK 4: MODEL EVALUATION & COMPARISON
# ==============================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

results = []
predictions = {}

# Evaluate each model on independent test data
for name, model in models.items():

    print("\n" + "=" * 50)
    print(name, "MODEL EVALUATION")
    print("=" * 50)

    # Predict probabilities
    probabilities = model.predict(X_test, batch_size=256)

    # Convert probabilities into class predictions
    y_pred = np.argmax(probabilities, axis=1)

    predictions[name] = y_pred

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test, y_pred,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        y_test, y_pred,
        average='macro',
        zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred,
        average='macro',
        zero_division=0
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

    # Print detailed classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # Plot confusion matrix
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3, 4]
    )

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES
    )

    plt.title(name + " Confusion Matrix")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.show()


# Create final comparison table
comparison = pd.DataFrame(results)

print("\nFINAL MODEL COMPARISON:")
print(comparison.round(4).to_string(index=False))

# Plot comparison of evaluation metrics
comparison_plot = comparison.set_index('Model')

comparison_plot.plot(
    kind='bar',
    figsize=(11, 5),
    ylim=(0, 1),
    rot=0
)

plt.title("RNN vs LSTM vs GRU Performance")
plt.ylabel("Score")
plt.xlabel("Model")
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()


# Compare training and validation accuracy
plt.figure(figsize=(12, 5))

for name, history in histories.items():

    plt.plot(
        history.history['accuracy'],
        label=name + ' Training'
    )

    plt.plot(
        history.history['val_accuracy'],
        linestyle='--',
        label=name + ' Validation'
    )

plt.title("Training and Validation Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

# Identify the best model based on macro F1-score
best_model = comparison.sort_values(
    by='F1 Score',
    ascending=False
).iloc[0]

print("\nBEST PERFORMING MODEL:")
print("Model:", best_model['Model'])
print("Accuracy:", round(best_model['Accuracy'], 4))
print("Precision:", round(best_model['Precision'], 4))
print("Recall:", round(best_model['Recall'], 4))
print("F1 Score:", round(best_model['F1 Score'], 4))


In [3]:

# ==============================
# TASK 4: MODEL EVALUATION & COMPARISON
# ==============================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

results = []
predictions = {}

# Evaluate each model on independent test data
for name, model in models.items():

    print("\n" + "=" * 50)
    print(name, "MODEL EVALUATION")
    print("=" * 50)

    # Predict probabilities
    probabilities = model.predict(X_test, batch_size=256)

    # Convert probabilities into class predictions
    y_pred = np.argmax(probabilities, axis=1)

    predictions[name] = y_pred

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test, y_pred,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        y_test, y_pred,
        average='macro',
        zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred,
        average='macro',
        zero_division=0
    )

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

    # Print detailed classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # Plot confusion matrix
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3, 4]
    )

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES
    )

    plt.title(name + " Confusion Matrix")
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.show()


# Create final comparison table
comparison = pd.DataFrame(results)

print("\nFINAL MODEL COMPARISON:")
print(comparison.round(4).to_string(index=False))

# Plot comparison of evaluation metrics
comparison_plot = comparison.set_index('Model')

comparison_plot.plot(
    kind='bar',
    figsize=(11, 5),
    ylim=(0, 1),
    rot=0
)

plt.title("RNN vs LSTM vs GRU Performance")
plt.ylabel("Score")
plt.xlabel("Model")
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()


# Compare training and validation accuracy
plt.figure(figsize=(12, 5))

for name, history in histories.items():

    plt.plot(
        history.history['accuracy'],
        label=name + ' Training'
    )

    plt.plot(
        history.history['val_accuracy'],
        linestyle='--',
        label=name + ' Validation'
    )

plt.title("Training and Validation Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

# Identify the best model based on macro F1-score
best_model = comparison.sort_values(
    by='F1 Score',
    ascending=False
).iloc[0]

print("\nBEST PERFORMING MODEL:")
print("Model:", best_model['Model'])
print("Accuracy:", round(best_model['Accuracy'], 4))
print("Precision:", round(best_model['Precision'], 4))
print("Recall:", round(best_model['Recall'], 4))
print("F1 Score:", round(best_model['F1 Score'], 4))


NameError: name 'models' is not defined